# Лабораторная работа по ТМО
## Фамилия Имя Отчество
## Группа ИУ5-_X_

#### Цель работы. Изучение основных методов анализа и прогнозирование временных рядов.

In [ ]:
# ============================================================
# Лабораторная работа по ТМО
# Анализ и прогнозирование временного ряда
# ============================================================

In [ ]:
# Запускать первой! После установки может потребоваться
# перезапуск среды (Runtime → Restart runtime), затем
# запустить все ячейки заново начиная с этой.

!pip install gplearn gmdhpy statsmodels scikit-learn pandas matplotlib

In [ ]:

import numpy as np
import pandas as pd
from matplotlib import pyplot
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
from gplearn.genetic import SymbolicRegressor
from gmdhpy.gmdh import MultilayerGMDH, RefFunctionType
import warnings

warnings.filterwarnings('ignore')

# Датасет вшит прямо в код — не нужен интернет и ничего скачивать не нужно.
# Ежемесячные продажи шампанского Perrin Freres, январь 1964 — сентябрь 1972.
# 105 наблюдений. Хорошо подходит для задачи прогнозирования:
#  - данные идут по месяцам;
#  - в данных ярко выражена сезонность (пики в декабре);
#  - наблюдается долгосрочный тренд роста продаж;
#  - объём достаточен для обучения нескольких моделей.

from io import StringIO

RAW_DATA = """Month,Sales
1964-01,2815
1964-02,2672
1964-03,2955
1964-04,3646
1964-05,3709
1964-06,3524
1964-07,3708
1964-08,3638
1964-09,3602
1964-10,3460
1964-11,5023
1964-12,7892
1965-01,3031
1965-02,2820
1965-03,3764
1965-04,3451
1965-05,3437
1965-06,3944
1965-07,4259
1965-08,3476
1965-09,3960
1965-10,3592
1965-11,5298
1965-12,8900
1966-01,3077
1966-02,2637
1966-03,3044
1966-04,4147
1966-05,4022
1966-06,3970
1966-07,4018
1966-08,3917
1966-09,4344
1966-10,4070
1966-11,6261
1966-12,9498
1967-01,2942
1967-02,2952
1967-03,3983
1967-04,4259
1967-05,4505
1967-06,3904
1967-07,4397
1967-08,4299
1967-09,4451
1967-10,4837
1967-11,6258
1967-12,9428
1968-01,3208
1968-02,3014
1968-03,3494
1968-04,4781
1968-05,4402
1968-06,4247
1968-07,4969
1968-08,4426
1968-09,4794
1968-10,4967
1968-11,6655
1968-12,9942
1969-01,3730
1969-02,3315
1969-03,4226
1969-04,4521
1969-05,4629
1969-06,4757
1969-07,4897
1969-08,4558
1969-09,4695
1969-10,5386
1969-11,7242
1969-12,9735
1970-01,4116
1970-02,3321
1970-03,4339
1970-04,4491
1970-05,5192
1970-06,5118
1970-07,5085
1970-08,5026
1970-09,5094
1970-10,5376
1970-11,7119
1970-12,10078
1971-01,3618
1971-02,3608
1971-03,4717
1971-04,4645
1971-05,5095
1971-06,4860
1971-07,5328
1971-08,4898
1971-09,5217
1971-10,5614
1971-11,7486
1971-12,10855
1972-01,3934
1972-02,3556
1972-03,4542
1972-04,5002
1972-05,4970
1972-06,5213
1972-07,5299
1972-08,5061
1972-09,5473
"""

ts = pd.read_csv(StringIO(RAW_DATA), header=0, index_col=0, parse_dates=True)
ts.columns = ['Sales']

display(ts.describe())

## 2. Визуализация временного ряда и его основных характеристик

### График временного ряда

In [ ]:
fig, ax = pyplot.subplots(1, 1, figsize=(12, 5))
ts.plot(ax=ax, legend=False)
ax.set_ylabel('Количество продаж')
ax.set_xlabel('Дата')
ax.set_title('Временной ряд: ежемесячные продажи шампанского')
pyplot.tight_layout()
pyplot.show()

### Первые 24 точки ряда (два года)

In [ ]:
fig, ax = pyplot.subplots(1, 1, figsize=(12, 5))
ts[:24].plot(ax=ax, legend=False)
ax.set_ylabel('Количество продаж')
ax.set_xlabel('Дата')
ax.set_title('Первые 24 наблюдения')
pyplot.tight_layout()
pyplot.show()

### Гистограмма распределения

In [ ]:
fig, ax = pyplot.subplots(1, 1, figsize=(10, 5))
ts.hist(ax=ax, bins=40, legend=False)
ax.set_xlabel('Количество продаж')
ax.set_ylabel('Частота')
ax.set_title('Гистограмма распределения значений')
pyplot.tight_layout()
pyplot.show()

### Оценка плотности распределения (KDE)

In [ ]:
fig, ax = pyplot.subplots(1, 1, figsize=(10, 5))
ts.plot(ax=ax, kind='kde', legend=False)
ax.set_xlabel('Количество продаж')
ax.set_title('Оценка плотности распределения')
pyplot.tight_layout()
pyplot.show()

### Автокорреляционная диаграмма

In [ ]:
fig, ax = pyplot.subplots(1, 1, figsize=(10, 5))
pd.plotting.autocorrelation_plot(ts, ax=ax)
ax.set_title('Автокорреляционная диаграмма')
pyplot.tight_layout()
pyplot.show()

## 3. Разделение временного ряда на обучающую и тестовую выборки

In [ ]:

# Целочисленная метка шкалы времени
xnum = list(range(ts.shape[0]))

# Разделение выборки: 70% обучение, 30% тест
Y = ts.values.ravel()
train_size = int(len(Y) * 0.7)
xnum_train, xnum_test = xnum[0:train_size], xnum[train_size:]
train, test = Y[0:train_size], Y[train_size:]

print(f'Общий размер ряда: {len(Y)} записей')
print(f'Обучающая выборка: {len(train)} записей '
      f'(с {ts.index[0].strftime("%Y-%m")} по {ts.index[train_size-1].strftime("%Y-%m")})')
print(f'Тестовая выборка:  {len(test)} записей '
      f'(с {ts.index[train_size].strftime("%Y-%m")} по {ts.index[-1].strftime("%Y-%m")})')
print(f'Соотношение train/test: {len(train)/len(Y)*100:.1f}% / {len(test)/len(Y)*100:.1f}%')

## 4. Прогнозирование временного ряда

### Метод 1: ARIMA (авторегрессионный метод)

In [ ]:

# ARIMA(5, 1, 1):
# p=5 — используем 5 предыдущих значений (авторегрессия)
# d=1 — дифференцируем ряд один раз (убираем тренд)
# q=1 — 1 лаг скользящего среднего

history_arima = [x for x in train]
arima_order = (5, 1, 1)

# Прогнозирование по одному шагу вперёд (walk-forward)
# На каждом шаге модель обучается заново, добавляя реальное значение
predictions_arima = []
for t in range(len(test)):
    model_arima = ARIMA(history_arima, order=arima_order)
    model_arima_fit = model_arima.fit()
    yhat_arima = model_arima_fit.forecast()[0]
    predictions_arima.append(yhat_arima)
    history_arima.append(test[t])

print(f'ARIMA: прогноз построен для {len(predictions_arima)} точек')

### Метод 2: Символьная регрессия (gplearn)

In [ ]:

# Создаём признаки из 7 лагов ряда
def create_lag_features(data, lags=7):
    df = pd.DataFrame(data, columns=['Sales'])
    for lag in range(1, lags + 1):
        df[f'lag{lag}'] = df['Sales'].shift(lag)
    df = df.dropna()
    X = df.drop('Sales', axis=1).values
    y = df['Sales'].values
    return X, y

X_train_gp, y_train_gp = create_lag_features(train, lags=7)
X_test_gp,  y_test_gp  = create_lag_features(test,  lags=7)
y_train_gp = y_train_gp.ravel()
y_test_gp  = y_test_gp.ravel()

# Обучение символьного регрессора
# Метод ищет математическую формулу через эволюционные алгоритмы
function_set = ['add', 'sub', 'mul', 'div', 'sin', 'cos']
est_gp = SymbolicRegressor(
    population_size=1500,
    metric='mse',
    generations=30,
    stopping_criteria=0.01,
    init_depth=(3, 8),
    verbose=1,
    function_set=function_set,
    const_range=(-50, 50),
    random_state=0
)
est_gp.fit(X_train_gp, y_train_gp)

# Прогноз на тестовой выборке
predictions_gp = est_gp.predict(X_test_gp)

print(f'\nНайденная формула:\n{est_gp._program}')

### Метод 3 и 4: МГУА (gmdhpy) — COMBI и MIA

In [ ]:

# Создаём лаговые признаки для МГУА (тот же принцип, что и для gplearn)
def create_lag_df(series, lags=7):
    df = pd.DataFrame(series, columns=['y'])
    for lag in range(1, lags + 1):
        df[f'lag{lag}'] = df['y'].shift(lag)
    return df.dropna()

df_train = create_lag_df(pd.Series(train), lags=7)
y_train_gmdh = df_train['y'].values
X_train_gmdh = df_train.drop('y', axis=1).values

# ── COMBI: линейный метод МГУА ──────────────────────────────
# Перебирает комбинации пар признаков, строит линейные нейроны.
# Отбор происходит по внешнему критерию (ошибка на валидации).

print('Обучение COMBI (линейный МГУА)...')
combi = MultilayerGMDH(ref_functions=RefFunctionType.rfLinear)
combi.fit(X_train_gmdh, y_train_gmdh)

predictions_combi = []
history_combi = [x for x in train]
for t in range(len(test)):
    last_7 = history_combi[-7:]
    if len(last_7) == 7:
        yhat_combi = combi.predict(np.array(last_7).reshape(1, -1))[0]
    else:
        yhat_combi = history_combi[-1]
    predictions_combi.append(yhat_combi)
    history_combi.append(test[t])

print('COMBI готов.')

# ── MIA: нелинейный метод МГУА ──────────────────────────────
# Использует нелинейные функции (квадратичные, кубические, произведения).
# Строит многослойную сеть из полиномиальных нейронов.

print('Обучение MIA (нелинейный МГУА)...')
mia = MultilayerGMDH(ref_functions=[
    RefFunctionType.rfLinear,
    RefFunctionType.rfLinearCov,
    RefFunctionType.rfQuadratic,
    RefFunctionType.rfCubic
])
mia.fit(X_train_gmdh, y_train_gmdh)

predictions_mia = []
history_mia = [x for x in train]
for t in range(len(test)):
    last_7 = history_mia[-7:]
    if len(last_7) == 7:
        yhat_mia = mia.predict(np.array(last_7).reshape(1, -1))[0]
    else:
        yhat_mia = history_mia[-1]
    predictions_mia.append(yhat_mia)
    history_mia.append(test[t])

print('MIA готова.')

## 5. Визуализация тестовой выборки и прогнозов

In [ ]:

# Создаём датафрейм с тестовыми данными и прогнозами
ts_test = ts.iloc[train_size:].copy()

# ARIMA: той же длины, что и test
ts_test['ARIMA'] = predictions_arima

# gplearn: короче на 7 точек из-за лагов — первые 7 = NaN
ts_test['GPLEARN'] = np.nan
gp_index = ts_test.index[7:]
ts_test.loc[gp_index, 'GPLEARN'] = predictions_gp

# COMBI и MIA: той же длины
ts_test['COMBI'] = predictions_combi
ts_test['MIA']   = predictions_mia

### График 1. Реальные данные и ARIMA

In [ ]:

fig, ax = pyplot.subplots(1, 1, figsize=(14, 6))
fig.suptitle('Прогноз ARIMA и реальные данные')
ts_test[['Sales', 'ARIMA']].plot(ax=ax, legend=True)
ax.set_ylabel('Продажи')
ax.set_xlabel('Дата')
pyplot.tight_layout()
pyplot.show()

### График 2. Реальные данные и символьная регрессия

In [ ]:

fig, ax = pyplot.subplots(1, 1, figsize=(14, 6))
fig.suptitle('Прогноз GPLEARN (символьная регрессия) и реальные данные')
ts_test[['Sales', 'GPLEARN']].plot(ax=ax, legend=True)
ax.set_ylabel('Продажи')
ax.set_xlabel('Дата')
pyplot.tight_layout()
pyplot.show()

### График 3. Реальные данные и COMBI с MIA

In [ ]:

fig, ax = pyplot.subplots(1, 1, figsize=(14, 6))
fig.suptitle('Прогнозы МГУА (COMBI и MIA) и реальные данные')
ts_test[['Sales', 'COMBI', 'MIA']].plot(ax=ax, legend=True)
ax.set_ylabel('Продажи')
ax.set_xlabel('Дата')
pyplot.tight_layout()
pyplot.show()

## 6. Оценка качества прогнозов

In [ ]:

# Расчёт метрик для каждого метода
rmse_arima = np.sqrt(mean_squared_error(test, predictions_arima))
mae_arima  = mean_absolute_error(test, predictions_arima)
print(f'ARIMA:    RMSE = {rmse_arima:.4f}, MAE = {mae_arima:.4f}')

# Символьная регрессия: выравниваем test по длине прогноза (смещение 7)
test_gp_aligned = test[7:]
rmse_gp = np.sqrt(mean_squared_error(test_gp_aligned, predictions_gp))
mae_gp  = mean_absolute_error(test_gp_aligned, predictions_gp)
print(f'GPLEARN:  RMSE = {rmse_gp:.4f}, MAE = {mae_gp:.4f}')

rmse_combi = np.sqrt(mean_squared_error(test, predictions_combi))
mae_combi  = mean_absolute_error(test, predictions_combi)
print(f'COMBI:    RMSE = {rmse_combi:.4f}, MAE = {mae_combi:.4f}')

rmse_mia = np.sqrt(mean_squared_error(test, predictions_mia))
mae_mia  = mean_absolute_error(test, predictions_mia)
print(f'MIA:      RMSE = {rmse_mia:.4f}, MAE = {mae_mia:.4f}')

# Сводная таблица
print(' ')
print('Сводная таблица качества прогноза')
print(' ')
print(f'{"Метод":<25} {"RMSE":>12} {"MAE":>12}')
print(f'{"ARIMA (5,1,1)":<25} {rmse_arima:>12.4f} {mae_arima:>12.4f}')
print(f'{"Символьная регрессия":<25} {rmse_gp:>12.4f} {mae_gp:>12.4f}')
print(f'{"COMBI (лин. МГУА)":<25} {rmse_combi:>12.4f} {mae_combi:>12.4f}')
print(f'{"MIA (нелин. МГУА)":<25} {rmse_mia:>12.4f} {mae_mia:>12.4f}')
print(' ')
print(f'Среднее значение ряда: {np.mean(Y):.4f}')
print(f'Стд. отклонение ряда:  {np.std(Y):.4f}')

# Столбчатые диаграммы сравнения RMSE и MAE
methods      = ['ARIMA', 'GPLEARN', 'COMBI', 'MIA']
rmse_values  = [rmse_arima, rmse_gp, rmse_combi, rmse_mia]
mae_values   = [mae_arima,  mae_gp,  mae_combi,  mae_mia]
colors       = ['#4472C4', '#ED7D31', '#A5A5A5', '#FFC000']

fig, axes = pyplot.subplots(1, 2, figsize=(14, 5))

axes[0].bar(methods, rmse_values, color=colors, edgecolor='black')
axes[0].set_title('Сравнение методов по RMSE')
axes[0].set_ylabel('RMSE')
for i, v in enumerate(rmse_values):
    axes[0].text(i, v + 0.02, f'{v:.0f}', ha='center', fontweight='bold')

axes[1].bar(methods, mae_values, color=colors, edgecolor='black')
axes[1].set_title('Сравнение методов по MAE')
axes[1].set_ylabel('MAE')
for i, v in enumerate(mae_values):
    axes[1].text(i, v + 0.02, f'{v:.0f}', ha='center', fontweight='bold')

fig.suptitle('Сравнение качества прогнозов', fontsize=14, fontweight='bold')
pyplot.tight_layout()
pyplot.show()

# Диаграммы рассеяния «Факт vs Прогноз»
fig, axes = pyplot.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

axes[0].scatter(test, predictions_arima, alpha=0.4, s=10, color='#4472C4')
axes[0].plot([test.min(), test.max()], [test.min(), test.max()], 'r--', lw=1)
axes[0].set_xlabel('Фактические значения')
axes[0].set_ylabel('Прогнозные значения')
axes[0].set_title(f'ARIMA (RMSE={rmse_arima:.0f})')

axes[1].scatter(test_gp_aligned, predictions_gp, alpha=0.4, s=10, color='#ED7D31')
axes[1].plot([test_gp_aligned.min(), test_gp_aligned.max()],
             [test_gp_aligned.min(), test_gp_aligned.max()], 'r--', lw=1)
axes[1].set_xlabel('Фактические значения')
axes[1].set_ylabel('Прогнозные значения')
axes[1].set_title(f'GPLEARN (RMSE={rmse_gp:.0f})')

axes[2].scatter(test, predictions_combi, alpha=0.4, s=10, color='#A5A5A5')
axes[2].plot([test.min(), test.max()], [test.min(), test.max()], 'r--', lw=1)
axes[2].set_xlabel('Фактические значения')
axes[2].set_ylabel('Прогнозные значения')
axes[2].set_title(f'COMBI (RMSE={rmse_combi:.0f})')

axes[3].scatter(test, predictions_mia, alpha=0.4, s=10, color='#FFC000')
axes[3].plot([test.min(), test.max()], [test.min(), test.max()], 'r--', lw=1)
axes[3].set_xlabel('Фактические значения')
axes[3].set_ylabel('Прогнозные значения')
axes[3].set_title(f'MIA (RMSE={rmse_mia:.0f})')

fig.suptitle('Диаграммы рассеяния: Фактические и прогнозные значения',
             fontsize=14, fontweight='bold')
pyplot.tight_layout()
pyplot.show()

## Вывод

In [ ]:

# В ходе работы были реализованы четыре метода прогнозирования временного ряда
# продаж шампанского (1964–1972).
#
# ARIMA(5,1,1) показала наилучший результат благодаря шаговому переобучению
# (walk-forward): на каждом шаге модель использует реальное предыдущее значение.
#
# Символьная регрессия (gplearn) сопоставима с ARIMA по RMSE, при этом
# строит явную математическую формулу — интерпретируемый результат.
#
# COMBI (линейный МГУА) показывает умеренную точность; линейная структура
# нейронов ограничивает способность захватить нелинейную сезонность.
#
# MIA (нелинейный МГУА) склонна к переобучению на малых выборках из-за
# большого числа параметров нелинейных нейронов (кубических и квадратичных).
#
# Результат типичен для временных рядов: простые специализированные методы
# (ARIMA) часто превосходят более сложные универсальные (MIA) на небольших данных.

print('Работа завершена.')